<a href="https://colab.research.google.com/github/iadicarlo/seavigil/blob/main/notebooks/sentinel1_vessel_detection.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# SeaVigil: Sentinel-1 vessel detection (Allen Institute model)

Runs the open, pre-trained **Allen Institute** Sentinel-1 detector (Apache-2.0, the model behind Skylight) on a Copernicus scene, on a free Colab GPU. Output is `seavigil_detections.csv`.

**Before you run:** (1) `Runtime` -> `Change runtime type` -> **GPU**. (2) key icon (left) -> add secrets `CDSE_S3_KEY` and `CDSE_S3_SECRET` (your Copernicus S3 keys), notebook access ON. (3) `Runtime` -> `Run all`.

Note: the model is a FastAPI service with a couple of repo path quirks; the cells below fix them (config path + model paths + cached backbones) and call it the supported way (a local server we POST to).


## 1. Clone the model + pull weights (Git LFS)


In [ ]:
!sudo apt-get -qq update >/dev/null && sudo apt-get -qq install -y git-lfs >/dev/null
!git lfs install >/dev/null
import os
if not os.path.isdir('/content/vessel-detection-sentinels'):
    !git clone --depth 1 https://github.com/allenai/vessel-detection-sentinels.git /content/vessel-detection-sentinels
%cd /content/vessel-detection-sentinels
!git lfs pull --include='data/model_artifacts/sentinel-1/**' --include='torch_weights/**'
!ls -la data/model_artifacts/sentinel-1/frcnn_cmp2/3dff445/best.pth data/model_artifacts/sentinel-1/attr/c34aa37/best.pth


## 2. Dependencies
Keeps Colab's GPU PyTorch (the model loads its weights into it); installs GDAL matched to the system lib plus the model's other deps and the server bits.


In [ ]:
# GDAL from apt (the pip wheel build is what fails); keep Colab's GPU PyTorch
!sudo apt-get -qq install -y gdal-bin libgdal-dev python3-gdal >/dev/null
!grep -ivE '^torch|^--extra-index|^gdal|^GDAL' requirements.txt > /tmp/reqs.txt
!pip -q install -r /tmp/reqs.txt uvicorn requests
import sys; sys.path.insert(0, '/usr/lib/python3/dist-packages')  # where apt puts osgeo
import torch; from osgeo import gdal
print('torch', torch.__version__, '| GPU:', torch.cuda.is_available(), '| GDAL', gdal.__version__)


## 3. Fix the repo's paths (config + weights + backbones)
The repo's config.yml points at Docker container paths and main.py looks for config one level too deep; we repoint it at the weights we just pulled and mirror it where the code expects.


In [ ]:
import yaml, os, shutil
ROOT = '/content/vessel-detection-sentinels'
cfg_path = f'{ROOT}/src/config/config.yml'
cfg = yaml.safe_load(open(cfg_path))
cfg['main']['sentinel1_detector']     = f'{ROOT}/data/model_artifacts/sentinel-1/frcnn_cmp2/3dff445'
cfg['main']['sentinel1_postprocessor'] = f'{ROOT}/data/model_artifacts/sentinel-1/attr/c34aa37'
yaml.safe_dump(cfg, open(cfg_path, 'w'))
# main.py builds CONFIG_PATH as <dir>/src/config/config.yml -> needs src/src/config/
os.makedirs(f'{ROOT}/src/src/config', exist_ok=True)
shutil.copy(cfg_path, f'{ROOT}/src/src/config/config.yml')
# cache the torchvision backbones where the model expects them (as the Dockerfile does)
os.makedirs('/root/.cache/torch/hub/checkpoints', exist_ok=True)
for w in ['resnet50-0676ba61.pth','swin_v2_s-637d8ceb.pth','swin_v2_t-b137f0e2.pth']:
    s = f'{ROOT}/torch_weights/{w}'
    if os.path.exists(s): shutil.copy(s, f'/root/.cache/torch/hub/checkpoints/{w}')
print('config repointed at pulled weights + mirrored; backbones cached')


## 4. Download a Sentinel-1 scene from Copernicus (S3)
Defaults to the Galapagos scene from 2026-06-27. For another reserve/date, find a scene at https://browser.dataspace.copernicus.eu (Sentinel-1, GRD, IW) and paste its `.SAFE` name + S3 date path.


In [ ]:
from google.colab import userdata
import boto3, pathlib
s3 = boto3.client('s3', endpoint_url='https://eodata.dataspace.copernicus.eu',
                  aws_access_key_id=userdata.get('CDSE_S3_KEY'), aws_secret_access_key=userdata.get('CDSE_S3_SECRET'))
SCENE = 'S1D_IW_GRDH_1SDV_20260627T114116_20260627T114145_003421_006075_6C48.SAFE'
S3_PREFIX = 'Sentinel-1/SAR/IW_GRDH_1S/2026/06/27/' + SCENE
dest = pathlib.Path(ROOT) / 'data' / SCENE
n = 0
for page in s3.get_paginator('list_objects_v2').paginate(Bucket='eodata', Prefix=S3_PREFIX):
    for obj in page.get('Contents', []):
        out = dest / obj['Key'][len(S3_PREFIX)+1:]
        out.parent.mkdir(parents=True, exist_ok=True)
        s3.download_file('eodata', obj['Key'], str(out))
        n += 1
print('downloaded', n, 'files ->', dest); get_ipython().system(f'du -sh {dest}')


## 5. Run inference (start the local server, then POST)
Starts the model's FastAPI server and posts the scene to it, the way the repo's own example does. The model loads and runs on the GPU; a scene takes a few minutes.


In [ ]:
import subprocess, time, requests
srv = subprocess.Popen(['python','main.py'], cwd=f'{ROOT}/src',
                       stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
ready = False
for _ in range(120):
    try:
        requests.get('http://localhost:5557/docs', timeout=2); ready = True; break
    except requests.exceptions.RequestException:
        if srv.poll() is not None:
            print('SERVER FAILED TO START:\n', srv.stdout.read()); raise SystemExit
        time.sleep(2)
print('server ready:', ready)
r = requests.post('http://localhost:5557/detections', timeout=3600, json={
    'scene_id': SCENE, 'raw_path': f'{ROOT}/data/', 'output_dir': f'{ROOT}/data/output',
    'conf': 0.85, 'nms_thresh': 10, 'save_crops': False, 'force_cpu': False})
print('detections response:', r.status_code, r.text[:400])
srv.terminate()


## 6. Review + download the detections
Then run `scripts/sar_detections_to_incidents.py --detections seavigil_detections.csv` in the SeaVigil repo to fold them into the map.


In [ ]:
import pandas as pd, glob
found = glob.glob(f'{ROOT}/data/output/**/predictions.csv', recursive=True)
if not found:
    print('No predictions.csv yet - check the server output above for errors.')
else:
    df = pd.read_csv(found[0]); df['scene_id'] = SCENE
    print(len(df), 'vessel detections in', SCENE)
    cols = [c for c in ['lon','lat','score','vessel_length_m','vessel_speed_k','is_fishing_vessel'] if c in df.columns]
    display(df[cols].head(25))
    df.to_csv('seavigil_detections.csv', index=False)
    from google.colab import files; files.download('seavigil_detections.csv')
